# Lesson 5: Grouping

- **Split-apply-combine** strategy
- `group_by()`
- Summary statistics
- Method chaining

In [1]:
import numpy as np
import pandas as pd

# Reading in the data
URL = 'https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv'
penguins = pd.read_csv(URL)

penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [7]:
penguins.rows

AttributeError: 'DataFrame' object has no attribute 'rows'

### Summary Statistics 
Callable methods for attaining summary statistics: 

- `sum()`: sum values in each column
- `count()`: count non-NA values in each column
- `size()`: count the number of values
- `min()` and `max()`: get the minimum and maxiumum values in each column
- `std()` and `var()`: get the standard deviation and variance in each column



In [5]:
# Getting the number of non-NA values in each column 
penguins.count()

species              344
island               344
bill_length_mm       342
bill_depth_mm        342
flipper_length_mm    342
body_mass_g          342
sex                  333
year                 344
dtype: int64

In [10]:
# Minimum value in each column with numerical values

penguins.select_dtypes('number').min() # Out of the penguins dataframe, selecting() dtypes that are numbers and getting the minimum values


bill_length_mm         32.1
bill_depth_mm          13.1
flipper_length_mm     172.0
body_mass_g          2700.0
year                 2007.0
dtype: float64

# Split-apply-combine

1. **Split:** Split the data into logical groups (e.g. species, sex, island, etc.)
2. **Apply:** Calculate some summary statistics on each group (e.g. average flipper length by species, number of individuals per island, body mass by index, etc.)
3. **Combine:** Combine the statistic calculated on each group together

For either pandas.df or pandas.Series, we can use the `group_by()` method to split/group the data into different categories.

General syntax for `group_by()` is:

`df.groupby(columns_to_group_by).summary_method()`

Typically, 'columns_to_group_by' is a single column name (string) or a list of column names separated by commas. Unique values 
of the column will be used as the groups of the dataframe (species names, etc.)




In [12]:
# Indexing the 'flipper_length_mm' column will return the mean of the values within the entire column. 
# This means all species, sexes, and categories are included. 
penguins['flipper_length_mm'].mean()

200.91520467836258

In [17]:
# Getting the mean flipper length by species
# Must add operation in order to receive an output 
penguins.groupby('species')['flipper_length_mm'].mean() 
# Explanation: using penguins df, we group by species, specify the flipper length column, take mean of indexed column with respect to group. 

species
Adelie       189.953642
Chinstrap    195.823529
Gentoo       217.186992
Name: flipper_length_mm, dtype: float64

In [19]:
# Creating new series 'mean_flipper_length' 
avg_flipper = (penguins.groupby('species')['flipper_length_mm'].mean().rename('mean_flipper_length').sort_values(ascending=False))
avg_flipper

species
Gentoo       217.186992
Chinstrap    195.823529
Adelie       189.953642
Name: mean_flipper_length, dtype: float64

In [22]:
# Grouping by combination of columns 
avg_flipper_species_island = (penguins.groupby(['species', 'island'])['flipper_length_mm'].mean().rename('avg_flipper_species_island').sort_values(ascending=False))
avg_flipper_species_island

species    island   
Gentoo     Biscoe       217.186992
Chinstrap  Dream        195.823529
Adelie     Torgersen    191.196078
           Dream        189.732143
           Biscoe       188.795455
Name: avg_flipper_species_island, dtype: float64

### Grouping and NA values 

We can check NAs by using two methods: 
- `size`: returns the number of observations in a `pandas.Series`
- `count()`: returns the number of non-NA observations in the series 

In [32]:
# Number of NA observations 
(penguins['sex'].size) - (penguins['sex'].count()) # subtracting the # of values in the series by the number of non-NA values

11

In [33]:
# By default, group_by() leaves out the rows where the grouping column has NA values. 

In [37]:
# Putting NA values as their own group 
penguins.groupby('sex', dropna=False).size()

sex
female    165
male      168
NaN        11
dtype: int64

### Check-in

In [38]:
penguins.groupby(['island', 'year']).count()

species  bill_length_mm  bill_depth_mm  flipper_length_mm  \
island    year                                                              
Biscoe    2007       44              44             44                 44   
          2008       64              64             64                 64   
          2009       60              59             59                 59   
Dream     2007       46              46             46                 46   
          2008       34              34             34                 34   
          2009       44              44             44                 44   
Torgersen 2007       20              19             19                 19   
          2008       16              16             16                 16   
          2009       16              16             16                 16   

                body_mass_g  sex  
island    year                    
Biscoe    2007           44   43  
          2008           64   63  
          2009           59   57  
Dream     2007           46   45  
          2008           34   34  
          2009           44   44  
Torgersen 2007           19   15  
          2008           16   16  
          2009           16   16

In [40]:
penguins.groupby(['island', 'year']).size()

island     year
Biscoe     2007    44
           2008    64
           2009    60
Dream      2007    46
           2008    34
           2009    44
Torgersen  2007    20
           2008    16
           2009    16
dtype: int64